# Exploraci?n y primer run de MLflow

Este cuaderno acompa?a al proyecto `mlflow-sueno`. Sirve para explorar el dataset y probar un run r?pido desde Jupyter.

Antes de ejecutarlo, fija el almac?n de MLflow y abre Jupyter desde la ra?z del proyecto `mlflow-sueno/`.


In [ ]:
import sys
sys.path.append("../src")

import mlflow
import pandas as pd

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("clasificacion-sueno")

from data import cargar_dataframe

df = cargar_dataframe()
print("Filas y columnas:", df.shape)
df.head()


## Distribuci?n de la variable objetivo

`sleep_debt_category` tiene cuatro clases. La clase `Severe` es menos del 1 % del dataset, as? que hay que mirar m?s que la accuracy.


In [ ]:
df["sleep_debt_category"].value_counts()


## Correlaciones con la fatiga del d?a siguiente

Aunque el modelo de este tutorial clasifica la deuda de sue?o, esta tabla ayuda a entender el dataset. Ojo: las correlaciones m?s fuertes son variables fisiol?gicas de la misma noche y **no** se usan como features para evitar fuga de informaci?n.


In [ ]:
df.select_dtypes("number").corr()["next_day_fatigue_score"].sort_values(ascending=False).round(2)


## Un run r?pido desde el notebook

Se reutiliza el mismo pipeline de `src/train.py` para que el preprocesamiento sea id?ntico al del entrenamiento principal.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from config import FEATURES, OBJETIVO, SEMILLA
from train import construir_modelo

X = df[FEATURES]
y = df[OBJETIVO]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y
)

with mlflow.start_run(run_name="notebook_baseline"):
    mlflow.log_params({"modelo": "RandomForestClassifier", "n_estimators": 100, "max_depth": 8})
    modelo = construir_modelo(100, 8, 2, "balanced_subsample").fit(X_train, y_train)
    pred = modelo.predict(X_test)
    metricas = {
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
    }
    mlflow.log_metrics(metricas)
    print(metricas)


## Siguiente paso

Para el flujo completo (registro, alias, serving y evaluaci?n) usa los scripts de `src/` como se explica en el `README.md`.
